# Task 7: Exploratory Data Analysis — Real DrugBank & ChEMBL Data
**Owner:** Sharise  
**Project:** CompoundIQ / PharmAI  
**Course:** ITAI 2376 Deep Learning  
**Data Sources:** ChEMBL (49,878 molecules), SIDER (side effects), STITCH (protein interactions)

---
## Overview
This EDA uses the **real datasets** from our CompoundIQ pipeline — not sample data.  
We analyze molecular properties, side effect distributions, and protein interaction patterns  
to understand the chemical space our models operate in.

## Step 1: Install & Import

In [ ]:
!pip install rdkit pyarrow pandas numpy matplotlib seaborn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries loaded")

## Step 2: Load Real Project Data

We load three datasets from the CompoundIQ pipeline:
- **ChEMBL**: 49,878 small molecules with computed properties
- **SIDER**: Side effect database (308K drug–side-effect pairs)
- **STITCH**: Protein interaction data (437K chemical–protein pairs)

**Upload the parquet files** from the project Google Drive before running.

In [ ]:
from google.colab import files, drive

# Option A: Mount Google Drive (if files are on Drive)
drive.mount('/content/drive')

# Update these paths to match your Drive location
CHEMBL_PATH = '/content/drive/MyDrive/chembl_processed_full.parquet'
SIDER_DETAIL_PATH = '/content/drive/MyDrive/sider_detailed.parquet'
SIDER_FEAT_PATH = '/content/drive/MyDrive/sider_features.parquet'
STITCH_DETAIL_PATH = '/content/drive/MyDrive/stitch_detailed.parquet'
STITCH_FEAT_PATH = '/content/drive/MyDrive/stitch_features.parquet'

# Load datasets
df_chembl = pd.read_parquet(CHEMBL_PATH)
df_sider = pd.read_parquet(SIDER_DETAIL_PATH)
df_sider_feat = pd.read_parquet(SIDER_FEAT_PATH)
df_stitch = pd.read_parquet(STITCH_DETAIL_PATH)
df_stitch_feat = pd.read_parquet(STITCH_FEAT_PATH)

print("=== DATASET SIZES ===")
print(f"ChEMBL molecules:        {len(df_chembl):>10,}")
print(f"SIDER side-effect pairs: {len(df_sider):>10,}")
print(f"SIDER drug features:     {len(df_sider_feat):>10,}")
print(f"STITCH interactions:     {len(df_stitch):>10,}")
print(f"STITCH drug features:    {len(df_stitch_feat):>10,}")

print(f"\n=== ChEMBL COLUMNS ===")
print(list(df_chembl.columns))
print(f"\n=== CHEMBL SAMPLE ===")
df_chembl.head()

## Visualization 1: Molecular Property Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

props = [
    ('molecular_weight', 'Molecular Weight (Da)', 'steelblue'),
    ('logp', 'LogP (Lipophilicity)', 'coral'),
    ('tpsa', 'TPSA (Polar Surface Area)', 'seagreen'),
    ('h_bond_donors', 'H-Bond Donors', 'mediumpurple'),
    ('h_bond_acceptors', 'H-Bond Acceptors', 'goldenrod'),
    ('num_rotatable_bonds', 'Rotatable Bonds', 'indianred'),
]

for ax, (col, title, color) in zip(axes.flatten(), props):
    data = df_chembl[col].dropna()
    ax.hist(data, bins=50, color=color, alpha=0.7, edgecolor='white')
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1.5, label=f'Median: {data.median():.1f}')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

fig.suptitle(f'Molecular Property Distributions — {len(df_chembl):,} ChEMBL Molecules', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_mol_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("Summary Statistics:")
print(df_chembl[['molecular_weight','logp','tpsa','h_bond_donors','h_bond_acceptors','num_rotatable_bonds']].describe().round(2))

## Visualization 2: Lipinski Rule of Five Analysis

Lipinski's Rule of Five predicts drug-likeness:
- MW ≤ 500 Da
- LogP ≤ 5
- H-bond donors ≤ 5
- H-bond acceptors ≤ 10

In [ ]:
df_lip = df_chembl.copy()
df_lip['lip_mw'] = df_lip['molecular_weight'] <= 500
df_lip['lip_logp'] = df_lip['logp'] <= 5
df_lip['lip_hbd'] = df_lip['h_bond_donors'] <= 5
df_lip['lip_hba'] = df_lip['h_bond_acceptors'] <= 10
df_lip['lip_violations'] = 4 - (df_lip['lip_mw'].astype(int) + df_lip['lip_logp'].astype(int) + df_lip['lip_hbd'].astype(int) + df_lip['lip_hba'].astype(int))
df_lip['is_druglike'] = df_lip['lip_violations'] <= 1

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Violations count
viol_counts = df_lip['lip_violations'].value_counts().sort_index()
axes[0].bar(viol_counts.index, viol_counts.values, color=['#2ecc71','#27ae60','#f39c12','#e74c3c','#c0392b'], edgecolor='white')
axes[0].set_title('Lipinski Violations per Molecule', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Violations')
axes[0].set_ylabel('Count')

# Drug-like pie
druglike = df_lip['is_druglike'].value_counts()
axes[1].pie(druglike.values, labels=['Drug-like (≤1 violation)', 'Non-drug-like (>1)'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 10})
axes[1].set_title('Drug-likeness Assessment', fontsize=12, fontweight='bold')

# Per-rule compliance
rules = ['MW ≤ 500', 'LogP ≤ 5', 'HBD ≤ 5', 'HBA ≤ 10']
compliance = [df_lip['lip_mw'].mean()*100, df_lip['lip_logp'].mean()*100,
              df_lip['lip_hbd'].mean()*100, df_lip['lip_hba'].mean()*100]
bars = axes[2].barh(rules, compliance, color=['steelblue','coral','seagreen','goldenrod'], edgecolor='white')
axes[2].set_xlim(0, 105)
for bar, val in zip(bars, compliance):
    axes[2].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=10)
axes[2].set_title('Per-Rule Compliance Rate', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_lipinski.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Drug-like molecules: {df_lip['is_druglike'].sum():,} / {len(df_lip):,} ({df_lip['is_druglike'].mean()*100:.1f}%)")

## Visualization 3: Side Effect Analysis (SIDER)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top 20 most common side effects
top_se = df_sider['side_effect'].value_counts().head(20)
axes[0].barh(range(len(top_se)), top_se.values, color='coral', edgecolor='white')
axes[0].set_yticks(range(len(top_se)))
axes[0].set_yticklabels(top_se.index, fontsize=8)
axes[0].invert_yaxis()
axes[0].set_title('Top 20 Most Common Side Effects (SIDER)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Drugs')

# Distribution of side effects per drug
se_per_drug = df_sider_feat['n_side_effects']
axes[1].hist(se_per_drug, bins=40, color='mediumpurple', alpha=0.7, edgecolor='white')
axes[1].axvline(se_per_drug.median(), color='black', linestyle='--', label=f'Median: {se_per_drug.median():.0f}')
axes[1].set_title(f'Side Effects per Drug — {len(df_sider_feat):,} Drugs', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Side Effects')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_sider.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Total unique side effects: {df_sider['side_effect'].nunique():,}")
print(f"Avg side effects per drug: {se_per_drug.mean():.1f}")
print(f"Max side effects for one drug: {se_per_drug.max()}")

## Visualization 4: Protein Interaction Network (STITCH)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Interaction confidence distribution
axes[0].hist(df_stitch['combined_score'], bins=50, color='teal', alpha=0.7, edgecolor='white')
axes[0].axvline(df_stitch['combined_score'].median(), color='black', linestyle='--', label=f"Median: {df_stitch['combined_score'].median():.0f}")
axes[0].set_title(f'Protein Interaction Confidence Scores — {len(df_stitch):,} Pairs', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Combined Confidence Score')
axes[0].set_ylabel('Count')
axes[0].legend()

# Interactions per chemical
int_per_chem = df_stitch_feat['n_protein_interactions']
axes[1].hist(int_per_chem, bins=50, color='darkorange', alpha=0.7, edgecolor='white')
axes[1].axvline(int_per_chem.median(), color='black', linestyle='--', label=f'Median: {int_per_chem.median():.0f}')
axes[1].set_title(f'Protein Interactions per Chemical — {len(df_stitch_feat):,} Chemicals', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Protein Interactions')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_stitch.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Unique chemicals: {df_stitch['chemical'].nunique():,}")
print(f"Unique proteins: {df_stitch['protein'].nunique():,}")
print(f"High confidence (>700): {(df_stitch['combined_score'] > 700).sum():,} ({(df_stitch['combined_score'] > 700).mean()*100:.1f}%)")

## Visualization 5: Correlation Heatmap

In [ ]:
numeric_cols = ['molecular_weight', 'logp', 'tpsa', 'h_bond_donors', 'h_bond_acceptors', 'num_rotatable_bonds', 'num_aromatic_rings', 'num_heavy_atoms', 'num_rings']
corr = df_chembl[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=1, ax=ax, vmin=-1, vmax=1)
ax.set_title(f'Property Correlation Heatmap — {len(df_chembl):,} Molecules', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Strongest correlations:")
corr_pairs = corr.unstack().drop_duplicates()
corr_pairs = corr_pairs[corr_pairs.abs() < 1].sort_values(ascending=False)
for idx, val in corr_pairs.head(5).items():
    print(f"  {idx[0]} vs {idx[1]}: {val:.3f}")

## Visualization 6: Chemical Space Overview (MW vs LogP)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

scatter = ax.scatter(df_chembl['molecular_weight'], df_chembl['logp'],
                     c=df_chembl['tpsa'], cmap='viridis', alpha=0.15, s=5, rasterized=True)
plt.colorbar(scatter, label='TPSA', shrink=0.8)

# Lipinski boundaries
ax.axvline(500, color='red', linestyle='--', alpha=0.7, label='MW = 500 (Lipinski)')
ax.axhline(5, color='red', linestyle='--', alpha=0.7, label='LogP = 5 (Lipinski)')

# Drug-like zone
from matplotlib.patches import Rectangle
rect = Rectangle((0, -5), 500, 10, fill=True, facecolor='green', alpha=0.05, label='Drug-like zone')
ax.add_patch(rect)

ax.set_xlabel('Molecular Weight (Da)', fontsize=11)
ax.set_ylabel('LogP', fontsize=11)
ax.set_title(f'Chemical Space: MW vs LogP — {len(df_chembl):,} Molecules', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(0, 700)
ax.set_ylim(-10, 15)

plt.tight_layout()
plt.savefig('eda_chemical_space.png', dpi=150, bbox_inches='tight')
plt.show()

in_zone = ((df_chembl['molecular_weight'] <= 500) & (df_chembl['logp'] <= 5)).sum()
print(f"Molecules in drug-like zone (MW≤500, LogP≤5): {in_zone:,} / {len(df_chembl):,} ({in_zone/len(df_chembl)*100:.1f}%)")

## Key Insights Summary

**Insight 1 — Drug-likeness:** The majority of our ChEMBL dataset falls within Lipinski's Rule of Five, confirming the molecules are drug-like candidates suitable for the JT-VAE generative model.

**Insight 2 — Side Effects:** SIDER data shows most drugs have dozens of reported side effects. This distribution informs our safety scoring module — rare side effects may be underrepresented.

**Insight 3 — Protein Interactions:** STITCH data reveals that most chemicals interact with multiple protein targets. The confidence score distribution shows many high-confidence interactions (>700), providing strong signal for the Three-Way GNN.

**Insight 4 — Feature Correlations:** Strong correlations between molecular weight, heavy atom count, and ring count suggest these are redundant features. The GNN benefits from the more independent features like LogP and TPSA.

**Insight 5 — Chemical Space:** The MW vs LogP scatter plot shows our dataset covers a wide chemical space with good density in the drug-like zone, giving the VAE a rich training distribution.